# Habitat Radiomics Survival Analysis (Habitat_MaxH1)

LASSO-Cox survival modeling of habitat sub-region radiomic features.
Set `SEL_SUBREGION` to choose the sub-region:
- 1 = H1 (necrotic core; the published **Habitat_MaxH1** score)
- 2 = H2 (transition zone)
- 3 = H3 (active rim)
- 'all' = merged (H1 + H2 + H3)

Input features (`features/rad_features_h{X}.csv`) are the max-aggregated habitat
sub-region radiomic features (PyRadiomics; see `config/pyradiomics_extraction.yaml`);
the upstream voxel-level K-means habitat masks (K=3, portal-venous phase) are
generated on the OneKey platform. Among the 12 candidate models
(max / mean / min aggregation x H1 / H2 / H3 / merged), Max-H1 was selected by the
highest cross-validation C-index.

## 1. Environment

In [ ]:
import os, yaml, pickle, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

with open('config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

label_file = config['label_file']
group_info = config['dataset_column']
event_col = config.get('event_col', 'event')
duration_col = config.get('duration_col', 'duration')
p_value = config.get('pvalue', 0.05)

# ===== Key parameter =====
SEL_SUBREGION = 1  # 1=H1, 2=H2, 3=H3, 'all'=merged. The published Habitat_MaxH1 score uses H1.
# =========================

subregion_name = 'Merged' if SEL_SUBREGION == 'all' else f'H{SEL_SUBREGION}'
task_dir = 'results/tasks/habitat_survival_merged' if SEL_SUBREGION == 'all' else f'results/tasks/habitat_survival_h{SEL_SUBREGION}'
os.makedirs(task_dir, exist_ok=True)
print(f'Subregion: {subregion_name}')
print(f'Output dir: {task_dir}')

## 2. Load data

In [ ]:
feature_file = 'features/rad_features.csv' if SEL_SUBREGION == 'all' else f'features/rad_features_h{SEL_SUBREGION}.csv'
rad_data = pd.read_csv(feature_file)
rad_data['ID'] = rad_data['ID'].astype(str)
print(f'Feature file: {feature_file}')
print(f'Samples: {len(rad_data)}, features: {len(rad_data.columns)-1}')

label_data = pd.read_csv(label_file)
label_data['ID'] = label_data['ID'].astype(str)
print(f'Events: {label_data[event_col].sum()}, censored: {(label_data[event_col]==0).sum()}')

## 3. Preprocessing

In [ ]:
from onekey_algo.custom.components.comp1 import normalize_df, select_feature

combined = pd.merge(rad_data, label_data[['ID', event_col, duration_col, group_info]], on='ID', how='inner')
data = normalize_df(combined, not_norm=['ID', event_col, duration_col], group=group_info, use_train=True).dropna(axis=1)

# Split by the predefined geographic/clinical cohort
train_data = data[data[group_info] == 'train'].copy()
test1_data = data[data[group_info] == 'test1'].copy()
test2_data = data[data[group_info] == 'test2'].copy()
print(f'Train: {len(train_data)}, Test1: {len(test1_data)}, Test2: {len(test2_data)}')

feature_cols = [c for c in data.columns if c not in ['ID', event_col, duration_col, group_info]]
filtered_features = select_feature(train_data[feature_cols].corr('pearson'),
                                   threshold=0.9,
                                   topn=2048,
                                   verbose=False)
print(f'Correlation filtering: {len(feature_cols)} -> {len(filtered_features)}')

## 4. Feature selection (univariable Cox -> LASSO-Cox)

In [ ]:
from onekey_algo.custom.components.survival import uni_cox, lasso_cox_cv, get_x_y_survival

# Univariable Cox screening
sel_features = uni_cox(train_data,
                      duration_col=duration_col,
                      event_col=event_col,
                      cols=filtered_features,
                      verbose=False,
                      topk=25,
                      pvalue_thres=p_value)

print(f'Univariable Cox significant features: {len(sel_features)}')

# LASSO-Cox feature selection
lasso_selected = sel_features  # fallback to univariable result
if len(sel_features) >= 2:
    train_subset = train_data[['ID'] + sel_features + [event_col, duration_col]].copy()
    X, y = get_x_y_survival(train_subset, val_outcome=1, event_col=event_col, duration_col=duration_col)

    lasso_result = lasso_cox_cv(X, y,
                               max_iter=100,
                               norm_X=False,
                               prefix='',
                               l1_ratio=0.5,
                               cv=10)

    lasso_selected = list(lasso_result.index)
    lasso_result.to_csv(f'{task_dir}/lasso_features.csv')
    display(lasso_result)

print(f'LASSO selected: {len(lasso_selected)} features')

## 5. Cox model

In [ ]:
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

cph = CoxPHFitter(penalizer=0.1)
cph.fit(train_data[lasso_selected + [duration_col, event_col]], duration_col=duration_col, event_col=event_col)
cph.print_summary()
cph.summary.to_csv(f'{task_dir}/cox_summary.csv')

# Risk score for all cohorts
train_data['risk_score'] = cph.predict_partial_hazard(train_data[lasso_selected])
test1_data['risk_score'] = cph.predict_partial_hazard(test1_data[lasso_selected])
test2_data['risk_score'] = cph.predict_partial_hazard(test2_data[lasso_selected])

# C-index
train_cindex = cph.concordance_index_
test1_cindex = concordance_index(test1_data[duration_col], -test1_data['risk_score'], test1_data[event_col])
test2_cindex = concordance_index(test2_data[duration_col], -test2_data['risk_score'], test2_data[event_col])
print(f'\nC-index: train={train_cindex:.4f}, test1={test1_cindex:.4f}, test2={test2_cindex:.4f}')

## 6. Risk stratification & Kaplan-Meier

In [ ]:
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

def find_cutoff(data, score_col, dur_col, evt_col):
    scores = data[score_col].values
    best_cut, best_p = np.median(scores), 1.0
    for cut in np.linspace(np.percentile(scores, 10), np.percentile(scores, 90), 100):
        h, l = data[data[score_col] >= cut], data[data[score_col] < cut]
        if len(h) < 10 or len(l) < 10 or h[evt_col].sum() < 3 or l[evt_col].sum() < 3:
            continue
        p = logrank_test(h[dur_col], l[dur_col], h[evt_col], l[evt_col]).p_value
        if p < best_p:
            best_p, best_cut = p, cut
    return best_cut, best_p

optimal_cutoff, _ = find_cutoff(train_data, 'risk_score', duration_col, event_col)
train_data['risk_group'] = np.where(train_data['risk_score'] >= optimal_cutoff, 'High', 'Low')
test1_data['risk_group'] = np.where(test1_data['risk_score'] >= optimal_cutoff, 'High', 'Low')
test2_data['risk_group'] = np.where(test2_data['risk_score'] >= optimal_cutoff, 'High', 'Low')
print(f'Optimal cutoff: {optimal_cutoff:.4f}')

In [ ]:
def format_pvalue(p):
    """Format a p-value for reporting."""
    if p < 0.001:
        return 'p < 0.001'
    elif p < 0.01:
        return f'p = {p:.3f}'
    else:
        return f'p = {p:.3f}'

def plot_km(data, dur_col, evt_col, grp_col, title, path):
    fig, ax = plt.subplots(figsize=(10, 8))
    kmf = KaplanMeierFitter()
    colors = {'High': '#E74C3C', 'Low': '#3498DB'}
    for g in ['High', 'Low']:
        gd = data[data[grp_col] == g]
        if len(gd) > 0:
            kmf.fit(gd[dur_col], gd[evt_col], label=f'{g} (n={len(gd)})')
            kmf.plot_survival_function(ax=ax, ci_show=True, color=colors[g])
    h, l = data[data[grp_col]=='High'], data[data[grp_col]=='Low']
    p_val = None
    if len(h) > 0 and len(l) > 0:
        p_val = logrank_test(h[dur_col], l[dur_col], h[evt_col], l[evt_col]).p_value
        p_text = format_pvalue(p_val)
        ax.set_title(f'{title}\nLog-rank {p_text}', fontsize=14)
    ax.set_xlabel('Time (months)'); ax.set_ylabel('Survival Probability')
    ax.legend(loc='lower left'); ax.set_ylim([0, 1.05]); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches='tight'); plt.show()
    return p_val

train_p = plot_km(train_data, duration_col, event_col, 'risk_group', f'Training Set ({subregion_name})', f'{task_dir}/km_train.png')
test1_p = plot_km(test1_data, duration_col, event_col, 'risk_group', f'Test Set 1 ({subregion_name})', f'{task_dir}/km_test1.png')
test2_p = plot_km(test2_data, duration_col, event_col, 'risk_group', f'Test Set 2 ({subregion_name})', f'{task_dir}/km_test2.png')

## 7. Forest plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, max(4, len(lasso_selected)*0.5)))
cph.plot(ax=ax)
ax.axvline(x=0, color='black', linestyle='--')
ax.set_xlabel('log(HR) with 95% CI'); ax.set_title(f'Forest Plot ({subregion_name})')
plt.tight_layout()
plt.savefig(f'{task_dir}/forest.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Save results

In [ ]:
# Save patient risk groups
pd.concat([
    train_data[['ID', 'risk_score', 'risk_group', duration_col, event_col]].assign(dataset='train'),
    test1_data[['ID', 'risk_score', 'risk_group', duration_col, event_col]].assign(dataset='test1'),
    test2_data[['ID', 'risk_score', 'risk_group', duration_col, event_col]].assign(dataset='test2')
]).to_csv(f'{task_dir}/patient_risks.csv', index=False)

# Save model
with open(f'{task_dir}/model.pkl', 'wb') as f:
    pickle.dump({'model': cph, 'features': lasso_selected, 'cutoff': optimal_cutoff,
                'train_cindex': train_cindex, 'test1_cindex': test1_cindex, 'test2_cindex': test2_cindex}, f)

print('=' * 50)
print('Survival analysis complete!')
print('=' * 50)
print(f'Subregion: {subregion_name}')
print(f'Selected features: {len(lasso_selected)}')
print(f'Train C-index: {train_cindex:.4f}')
print(f'Test1 C-index: {test1_cindex:.4f}')
print(f'Test2 C-index: {test2_cindex:.4f}')
if train_p: print(f'Train Log-rank {format_pvalue(train_p)}')
if test1_p: print(f'Test1 Log-rank {format_pvalue(test1_p)}')
if test2_p: print(f'Test2 Log-rank {format_pvalue(test2_p)}')
print(f'\nResults saved to: {task_dir}')